### Comparison of BotSort and Bytetack Models for Robot Tracking

From the results below, I decided to move forward with Botsort! Botsort had a higher detection per frame as well as accounted for less unique IDs. These metrics are ideal because we want to identify the 6 robots during a match.

In [ ]:
import os
from pathlib import Path

# Assigning paths 
REPO_ROOT = Path(os.getcwd()).parent  
MODEL_PATH = REPO_ROOT / "yolov8_model" / "yolov8s.pt"
VIDEO_DIR  = REPO_ROOT / "data_source" / "videos"

In [2]:
# Finding path to best model
for p in REPO_ROOT.rglob("best_tuned_yolov8.pt"):
    print(p)

/work/classtmp/kmwendl/FIRST-Robotics-Competition-Data-Challenge/yolov8_model/best_tuned_yolov8.pt


In [ ]:
# Choosing random video, this can be changed
VIDEO_NAME = 'cropped_Qualification 45 - 2025 Central Missouri Regional.mp4'
VIDEO_PATH = "/work/classtmp/FIRST-Robotics-Competition-Data-Challenge-Videos/cropped_videos/cropped_Qualification 45 - 2025 Central Missouri Regional.mp4"

# Change Model Path to be best model from above in line 2!
MODEL_PATH = REPO_ROOT / "yolov8_model" / "best_tuned_yolov8.pt"

# Custom tracker, this is where I adjusted parameters 
CUSTOM_TRACKER_PATH = REPO_ROOT / "trackers"

# Checking to see if files exist! 
print("Model:", MODEL_PATH)
print("Video:", VIDEO_PATH)
print("Tracker:", CUSTOM_TRACKER_PATH)

print("Model exists:", MODEL_PATH.exists())
# print("Video exists:", VIDEO_PATH.exists())
# print("Tracker exists:", CUSTOM_TRACKER_PATH.exists())

Model: /work/classtmp/kmwendl/FIRST-Robotics-Competition-Data-Challenge/yolov8_model/best_tuned_yolov8.pt
Video: /work/classtmp/FIRST-Robotics-Competition-Data-Challenge-Videos/cropped_videos/cropped_Qualification 45 - 2025 Central Missouri Regional.mp4
Tracker: /work/classtmp/kmwendl/FIRST-Robotics-Competition-Data-Challenge/trackers
Model exists: True


In [ ]:
from ultralytics import YOLO
from collections import defaultdict
import numpy as np
import cv2

# Runs the BotSort and Bytetrack algorithms
def run_tracker(model_path, video_path, tracker_name, save_video=False, output_dir=None):
    
    print(f"\nRunning {tracker_name}...")
    model = YOLO(model_path)

    # BotSort model
    results = model.track(
        source=str(video_path),
        tracker=f"{tracker_name}.yaml",
        stream=True,
        device=0,         # Utilizing GPU! Faster processing 
        persist=False,    # Maintains ID across frames! 
        verbose=False
    )

    track_ids_per_frame = []
    unique_ids = set()
    id_lifetimes = defaultdict(int)

    video_writer = None

    for result in results:
        if result.boxes.id is not None:
            ids = result.boxes.id.cpu().numpy().astype(int)
            track_ids_per_frame.append(len(ids))

            for tid in ids:
                unique_ids.add(tid)
                id_lifetimes[tid] += 1
        else:
            track_ids_per_frame.append(0)

        if save_video:                   # Runs seperately to save off videos!
            frame = result.plot()

            if video_writer is None:
                h, w = frame.shape[:2]
                output_path = output_dir / f"{tracker_name}_test.mp4"

                video_writer = cv2.VideoWriter(
                    str(output_path),
                    cv2.VideoWriter_fourcc(*"mp4v"),    # Changed to mp4 
                    30,
                    (w, h)
                )

            video_writer.write(frame)

    if video_writer:
        video_writer.release()

    avg_per_frame   = np.mean(track_ids_per_frame) if track_ids_per_frame else 0
    avg_lifetime    = np.mean(list(id_lifetimes.values())) if id_lifetimes else 0

    print(f"{tracker_name} done!")
    return {
        'tracker'                   : tracker_name,
        'unique_track_ids'          : len(unique_ids),         
        'avg_detections_per_frame'  : round(avg_per_frame, 2),
        'avg_track_lifetime_frames' : round(avg_lifetime, 2),
        'total_detections'          : sum(track_ids_per_frame)
    }

In [ ]:
output_dir = REPO_ROOT / "tracking_output"
output_dir.mkdir(exist_ok=True)

# Runs byterack model
bytetrack_stats = run_tracker(
    MODEL_PATH, 
    VIDEO_PATH, 
    "bytetrack",
    save_video=False,
    output_dir=output_dir
)

# Runs botsort model
botsort_stats = run_tracker(
    MODEL_PATH, 
    VIDEO_PATH, 
    "botsort",
    save_video=False,
    output_dir=output_dir
)


Running bytetrack...
bytetrack done!

Running botsort...
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
botsort done!


In [ ]:
# Comparison table! 
print(f"{'Metric':<35} {'Bytetrack':>12} {'BotSort':>12}")
print("-" * 60)

metrics = ['total_detections', 'unique_track_ids', 'avg_detections_per_frame', 'avg_track_lifetime_frames']
friendly = ['Total Detections', 'Unique Track IDs', 'Avg Detections/Frame', 'Avg Track Lifetime (frames)']

for metric, name in zip(metrics, friendly):
    print(f"{name:<35} {str(bytetrack_stats[metric]):>12} {str(botsort_stats[metric]):>12}")

Metric                                 Bytetrack      BotSort
------------------------------------------------------------
Total Detections                           37378        37486
Unique Track IDs                             369          352
Avg Detections/Frame                        6.55         6.56
Avg Track Lifetime (frames)                101.3       106.49
